<div style="text-align: right;">
  <img src="https://raw.githubusercontent.com/exasol/ai-lab/refs/heads/main/assets/Exasol_Logo_2025_Dark.svg" style="width:200px; margin: 10px;" />
</div>

# Text classification model

In this notebook, we will load and use a text classification language models that can assign a label to a given text. Learn more about the Text Classification task <a href="https://huggingface.co/tasks/text-classification" target="_blank" rel="noopener">here</a>. Please also refer to the Transformer Extension <a href="https://github.com/exasol/transformers-extension/blob/main/doc/user_guide/user_guide.md" target="_blank" rel="noopener">User Guide</a> to find more information about the UDFs used in this notebook.

There are multiple UDfs for text_classification. 
The `AI_SENTIMENT` udf uses some 
default configuration parameters and a preselected model to find the sentiment of a given input text. 
If you need more control over your prediction, the `AI_CUSTOM_CLASSIFY_EXTENDED` udf allows you to set all configuration parameters. This allows you to use different models, which might sort the input into different categories.
We also have the `AI_ENTAILMENT_EXTENDED` udf, which allows you to select your own model and use it to judge how closley related two text fragments are.

After you finish the setup, we will take a closer look at each of the available UDFs.

We will be running SQL queries using <a href="https://github.com/ploomber/jupysql" target="_blank" rel="noopener">JupySQL</a>SQL Magic.

## Prerequisites

Prior to using this notebook the following steps need to be completed:
1. [Configure the AI Lab](../main_config.ipynb).
2. [Initialize the Transformer Extension](te_init.ipynb).

## Setup

### Open Secure Configuration Storage

In [ ]:
from exasol.nb_connector.ui.access import access_store
display(access_store.get_access_store('../'))

Let's bring up JupySQL and connect to the database via SQLAlchemy. 
Please refer to the documentation of [sqlalchemy-exasol](https://github.com/exasol/sqlalchemy-exasol) for details on how to connect to the database using the Exasol SQLAlchemy driver.

In [ ]:
from exasol.nb_connector.ui.common import jupysql
jupysql.init(ai_lab_config)

# AI_SENTIMENT

The `AI_SENTIMENT` udf uses some the [tabularisai/robust-sentiment-analysis](https://huggingface.co/tabularisai/robust-sentiment-analysis) model to find the sentiment(positive or negative) of a given input text.

Let's try to classify a single "product review": "These prediction UDFs are fantastic!".
We will save the result in the variable `udf_output` to support automatic testing of this notebook.

In [ ]:
%%sql --save udf_output
WITH MODEL_OUTPUT AS
(
    SELECT AI_SENTIMENT(
        'These prediction UDFs are fantastic!',
    )
)
SELECT label, score, error_message FROM MODEL_OUTPUT

The code above shows how the model works on a toy example. However, the main purpose of having a model deployed in the database is to get a quick response for a batch input. The performance gain comes from two factors - localization and parallelization. The first means that the input data never crosses the machine boundaries. The second means that multiple instances of the model are processing the data on all available nodes in parallel.

Another advantage of making predictions within the database is enhanced data security. The task of safeguarding privacy can be simplified given the fact that the source data never leaves the database machine.

In a more practical application, the text to be classified would be stored in a column of a database table. For example, if we wanted to get a label with the highest score for each row of the input table `MY_TEXT_TABLE`, where the text in question is in the column `MY_TEXT_COLUMN`, the SQL would look similar to this:
```
SELECT AI_SENTIMENT(MY_TEXT_COLUMN) FROM MY_TEXT_TABLE;
```
Please note, that the response time observed on the provided example with a single input will not be scaled up linearly in case of multiple inputs. Much of the latency falls on loading the model into the CPU memory from BucketFS. This needs to be done only once regardless of the number of inputs.
    
This is also True for the UDFs shown below.

# AI_CUSTOM_CLASSIFY_EXTENDED

If you want to use a different model for prediction, you can use the `AI_CUSTOM_CLASSIFY_EXTENDED` udf instead. This will work nearly the same, but you need to have a model installed and then set the udf parameters so the model can be found by the udf.

## Get a language model

To demonstrate the text classification task we will use the [Ekman emotions classifier](https://huggingface.co/arpanghoshal/EkmanClassifier) model.

We need to load the model from the Hugging Face Hub into the [BucketFS](https://docs.exasol.com/db/latest/database_concepts/bucketfs/bucketfs.htm). This could potentially be a long process, depending on the speed of the database connection. Unfortunately, we cannot tell exactly when it has finished. The notebook's hourglass may not be a reliable indicator. BucketFS will still be doing some work when the call issued by the notebook returns. Please wait for a few moments after that, before querying the model.

You might see a warning that some weights are newly initialized and the model should be trained on a down-stream task. Please ignore this warning. For the purpose of this demonstration, it is not important, the model should still be able to produce some meaningful output.

In [ ]:
from exasol.nb_connector.model_installation import install_model, TransformerModel
from transformers import AutoModelForSequenceClassification

# This is the name of the model at the Hugging Face Hub
MODEL_NAME = 'arpanghoshal/EkmanClassifier'
install_model(ai_lab_config, TransformerModel(MODEL_NAME, 'text_classification', AutoModelForSequenceClassification))

## Use the language model

Additionally to the input text, the `AI_CUSTOM_CLASSIFY_EXTENDED` takes the following parameters:

* device_id: To run the UDF on a GPU, specify the valid cuda device ID.
* bucketfs_conn: The BucketFS connection name.
* sub_dir: The directory where the model is stored in the BucketFS.
* model_name: The name of the model to use for prediction.
* return_ranks: Either "ALL" or "HIGHEST".
You need to supply these parameters in the correct order. Further information can be found in the  <a href="https://github.com/exasol/transformers-extension/blob/main/doc/user_guide/user_guide.md" target="_blank" rel="noopener">User Guide</a>.

Let's try to classify a single phrase that definitely bears emotions but is also somewhat ambiguous - "Oh my God!".
We will save the result in the variable `udf_output` to support automatic testing of this notebook.

In [ ]:
%%sql --save udf_output
WITH MODEL_OUTPUT AS
(
    SELECT AI_CUSTOM_CLASSIFY_EXTENDED(
        NULL,
        '{{ai_lab_config.bfs_connection_name}}',
        '{{ai_lab_config.bfs_model_subdir}}',
        '{{MODEL_NAME}}',
        'Oh my God!',
        'HIGHEST'
    )
)
SELECT label, score, rank, error_message FROM MODEL_OUTPUT

In this instance, we let the model return only the highest ranking result by setting `return_ranks = "HIGHEST"`. If we want to get all available results instead, we can run it with `return_ranks = "ALL"` instead:

In [ ]:
%%sql --save udf_output
WITH MODEL_OUTPUT AS
(
    SELECT AI_CUSTOM_CLASSIFY_EXTENDED(
        NULL,
        '{{ai_lab_config.bfs_connection_name}}',
        '{{ai_lab_config.bfs_model_subdir}}',
        '{{MODEL_NAME}}',
        'Oh my God!',
        'ALL'
    )
)
SELECT label, score, rank, error_message FROM MODEL_OUTPUT  ORDER BY SCORE DESC

We select only some of the udf's output columns in these examples.  If you need more details, you can find information on all available output columns in the <a href="https://github.com/exasol/transformers-extension/blob/main/doc/user_guide/user_guide.md" target="_blank" rel="noopener">User Guide</a>.

The output of the model is sorted into the following columns by the udf:

* label: the label the model assigned for the input
* score: the confidence, with which the label was assigned
* rank: the rank of the label. In this context, all predictions/labels for one input are ranked by their score. rank=1 means best result/highest score.
* error_message: error occurring while executing the udf will be saved here

### AI_ENTAILMENT_EXTENDED

The `AI_ENTAILMENT_EXTENDED` UDF uses the same parameters as the `AI_CUSTOM_CLASSIFY_EXTENDED` UDF, plus an aditional input text.

Now we are going to add some context to our exclamation in the form of an additional input text, and use the `AI_ENTAILMENT_EXTENDED` UDF to analyze our text again. Let's see how it will change the model output.

In [ ]:
%%sql --save udf_output
WITH MODEL_OUTPUT AS
(
    SELECT AI_ENTAILMENT_EXTENDED(
        NULL,
        '{{ai_lab_config.bfs_connection_name}}',
        '{{ai_lab_config.bfs_model_subdir}}',
        '{{MODEL_NAME}}',
        'Oh my God!',
        'I lost my purse.',
        'ALL'
    )
)
SELECT label, score, rank, error_message FROM MODEL_OUTPUT ORDER BY SCORE DESC